In [20]:
"""
====================================================
ERA5 Visualization Script
====================================================
"""

'\n====================================================\nERA5 Visualization Script\n====================================================\n'

In [21]:
#######################
#DIRECTORIES

In [22]:
# #SETTING UP DIRECTORIES
# mainDirectory = '/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/Regional-MPAS-Project/'
# workingDirectory="/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/Regional-MPAS-Project/DataAnalysis/InputData_DataAnalysis/"
# print(workingDirectory)
# outputDirectory=workingDirectory+"OUTPUT/"
# dataDirectory=mainDirectory+"DownloadData/DATA/ERA5_Data/"

In [23]:
#SETTING UP DIRECOTRIES
mainDirectory = "/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/"
codeDirectory = "/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/"
def SetOutputDirectory(campaign):
    import os
    outputDirectory=codeDirectory+"OUTPUT/DataAnalysis/ERA5_Data"
    outputDirectory=os.path.join(outputDirectory, campaign)
    os.makedirs(outputDirectory, exist_ok=True)
    return outputDirectory

def SetDataDirectory(campaign):
    import os
    dataDirectory=codeDirectory+"DATA/ERA5_Data/"
    dataDirectory=os.path.join(dataDirectory, campaign)
    os.makedirs(dataDirectory, exist_ok=True)
    return dataDirectory

In [24]:
#######################
#LIBRARIES, FUNCTIONS, and CLASSES

In [25]:
#IMPORT LIBRARIES
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Libraries/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "Libraries",
]

for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [26]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Functions_2.0/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [27]:
#IMPORT CLASSES
# --- Add your Functions folder to sys.path ---
import sys
path = mainDirectory + 'Functions_2.0/Classes/'
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "Classes_InputData_DataAnalysis",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [28]:
###########################
#FUNCTIONS

In [29]:
#MAKE DATE FOLDER (for output) FUNCTION
def MakeDateFolder(date_string):
    date_folder = strings.DateString(date_string)
    #adding date to output folder
    subdir = os.path.join(outputDirectory, date_folder)
    os.makedirs(subdir, exist_ok=True)
    return date_folder

def FixVariableName(variable,var_data):
    if variable == 'divergence':
        variable = 'convergence'
    return variable,var_data

def GetLoadDirectory(dataDirectory,date_folder):
    loadDirectorys = [
        os.path.join(dataDirectory, date_folder, f"{var}_ERA5_{date_folder}.nc")
        for var in variables.keys()
    ]
    return loadDirectorys

In [30]:
#RUN CALCULATIONS and PLOTTING #*#*#*#*#*#*# (this version adds quiver to U/V plots)
def RunCalculations(numerics, var_data, units, variable, calculation, mult_factor):
    calculation_results = {}

    arr   = var_data
    units = units
    if mult_factor != "NaN":
        arr *= mult_factor

    tz, _ = Ultimate_AreaAverage(var_data, dims=('t','z','y','x'), dim_names=('t','z'), mode='keep')
    t, _  = Ultimate_AreaAverage(tz,  dims=('t','z'),        dim_names=('t',),   mode='keep')

    three_hours = 3 * numerics.hour_index
    tz_3h   = calculation.block_vertical_profiles_2D(tz,  block=three_hours)
    tzyx_3h = calculation.block_vertical_profiles_4D(arr, block=three_hours)

    calculation_results[variable] = {
        "units": units,
        "tz": tz,          # (t,z)
        "t": t,            # (t,)
        "tz_3h": tz_3h,    # (nblocks, z)
        "tzyx_3h": tzyx_3h # (nblocks, z, y, x)
    }

    return calculation_results

def RunPlots(numerics, calculation_results, date_string, UTC_offset, outputFile, plotting, 
             colormap, vline, data_lim, line_contour, center_contour):
    for name, result in calculation_results.items():
        common_args = {
            "var_name": name,
            "var_units": result["units"],
            "date_string": date_string,
            "date_folder":  date_folder,
            "outputFile": outputFile,
            "numerics": numerics,
            "data_lim": data_lim,
            "UTC_offset": UTC_offset,
        }

        plotting.TZContourPlot(var_data=result['tz'], colormap=colormap, center_contour=center_contour, **common_args)
        plotting.TimeSeries(var_data=result['t'], **common_args)
        plotting.MultiAverage_VerticalProfiles(var_data=result['tz_3h'], vline=vline, **common_args)
        plotting.MultiAverage_HorizontalFields(var_data=result['tzyx_3h'], plev=1000, line_contour=line_contour, colormap=colormap, center_contour=center_contour, **common_args)
        if name in ['specific_cloud_liquid_water_content','specific_cloud_ice_water_content','specific_rain_water_content']:
            plotting.MultiAverage_HorizontalFields_VerticalAvg(var_data=result['tzyx_3h'], line_contour=line_contour, 
                                                               colormap=colormap, center_contour=center_contour, **common_args)

#RUNNING CALCULATIONS FUNCTION
def RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting):
    # global variable,calculation_results_temp1,calculation_results_temp2 #*#* quiver
    for count, (loadDirectory, (variable, components)) in enumerate(tqdm(zip(loadDirectorys, variables.items()), total=len(loadDirectorys), desc="Running Calculations"),start=1):
        units, colormap, vline, mult_factor = components["unit"], components["colormap"], components["vline"], components["mult_factor"]
        data_lim, line_contour, center_contour = components["data_lim"], components["line_contour"], components["center_contour"]
        
        #print
        print(f"Plotting {len(variables)} Variables",'\n')
        print(f"{count}. {variable} ({units}) → {loadDirectory}")
        
        #loading the variable
        ncFile=xr.open_dataset(loadDirectory)
        var_name = [v for v in list(ncFile.data_vars) if v not in ["number", "expver"]][0]
        # print('\n',var_name,'***')
        var_data=ncFile[var_name].data
        [variable,var_data] = FixVariableName(variable,var_data)
        numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,
                           TIME=ncFile[var_name]['valid_time'].data,P=ncFile[var_name]['pressure_level'].data,LAT=ncFile[var_name]['latitude'].data,LON=ncFile[var_name]['longitude'].data)
        print(variable+":\n","\t(Nt, Np, Nlat, Nlon) = ",(numerics.Nt,numerics.Np,numerics.Nlat,numerics.Nlon),"\n")
    
        #making output filename
        outputFile = os.path.join(outputDirectory, date_folder, variable) #variable also can be var_name
        
        os.makedirs(outputFile, exist_ok=True)
    
        #doing calculations
        calculation_results=RunCalculations(numerics, var_data, "("+units+")", variable, calculation, mult_factor)
        # ##### #*#* quiver
        # if variable in ["u_component_of_wind","v_component_of_wind"]:
        #     calculation_results_temp1=RunCalculations(numerics, var_data, "("+units+")", "u_component_of_wind", calculation, mult_factor)
        #     calculation_results_temp2=RunCalculations(numerics, var_data, "("+units+")", "v_component_of_wind", calculation, mult_factor)
        # #####
    
        #plotting
        RunPlots(numerics, calculation_results, date_string, UTC_offset, outputFile, plotting, 
                 colormap, vline, data_lim, line_contour, center_contour)

In [31]:
###########################
#LOADING DATA

In [32]:
#load in ERA5 data
variables = {
    "u_component_of_wind": {"unit": r"$m\ s^{-1}$", "colormap": "RdBu_r", "vline": 0, "mult_factor": "NaN", "data_lim": "NaN","line_contour": "F", "center_contour": 0,},
    "v_component_of_wind": {"unit": r"$m\ s^{-1}$", "colormap": "RdBu_r", "vline": 0, "mult_factor": "NaN", "data_lim": "NaN","line_contour": "F","center_contour": 0,},
    "vertical_velocity": {"unit": r"$Pa\ s^{-1}$", "colormap": "RdBu_r", "vline": 0, "mult_factor": "NaN", "data_lim": "NaN","line_contour": "F","center_contour": 0,},
    "divergence": {"unit": r"$s^{-1}$", "colormap": "RdBu_r", "vline": 0, "mult_factor": -1, "data_lim": "NaN","line_contour": "F","center_contour": 0,}, #mult_factor: converts to convergence
    "vorticity": {"unit": r"$s^{-1}$", "colormap": "RdBu_r", "vline": 0, "mult_factor": "NaN", "data_lim": "NaN","line_contour": "F","center_contour": 0,},
    "temperature": {"unit": r"$K$", "colormap": "coolwarm", "vline": "NaN", "mult_factor": "NaN", "data_lim": "NaN","line_contour": "F","center_contour": "NaN",},
    "specific_humidity": {"unit": r"$g\ kg^{-1}$", "colormap": "YlGnBu", "vline": 0, "mult_factor": 1e3, "data_lim": "NaN","line_contour": "F","center_contour": "NaN",},
    "specific_cloud_liquid_water_content": {"unit": r"$g\ kg^{-1}$", "colormap": "Blues", "vline": 0, "mult_factor": 1e3, "data_lim": "NaN","line_contour": "F","center_contour": "NaN",},
    "specific_cloud_ice_water_content": {"unit": r"$g\ kg^{-1}$", "colormap": "Purples", "vline": 0, "mult_factor": 1e3, "data_lim": "NaN","line_contour": "F","center_contour": "NaN",},
    "specific_rain_water_content": {"unit": r"$g\ kg^{-1}$", "colormap": "GnBu", "vline": 0, "mult_factor": 1e3, "data_lim": "NaN","line_contour": "F","center_contour": "NaN",},
    "relative_humidity": {"unit": r"$\%$", "colormap": "BrBG", "vline": 0, "mult_factor": "NaN", "data_lim": (0,100),"line_contour": "F","center_contour": "NaN",},
    "geopotential": {"unit": r"$m^{2}\ s^{-2}$", "colormap": "viridis", "vline": "NaN", "mult_factor": "NaN", "data_lim": "NaN","line_contour": "T","center_contour": "NaN",}
}

In [33]:
###########################
#RUNNING

In [15]:
##########################################################
# DOWNLOADING TRACER CAMPAIGN DATA
##########################################################

In [32]:
# coorindates information
# Data BOUNDING BOX centered at Houston, TX Mobile Facility (TRACER) Facility S2 ==> CSAP (C-Band Scanning ARM Precipitation Radar)
# (29.532N, 95.284W)
outputDirectory=SetOutputDirectory(campaign="TRACER")
dataDirectory=SetDataDirectory(campaign="TRACER")
#UTC OFFSET
UTC_offset="-5"

In [ ]:
###########################
#DATE ONE (DRY CASE)

#date information
date_string = "06-08 - 06-10 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

In [84]:
###########################
#DATE TWO (MOIST CASE)

#date information
date_string = "06-30 - 07-02 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/1 [00:00<?, ?it/s]/glade/derecho/scratch/aroseman/tmp/ipykernel_4486/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 1 Variables 

1. u_component_of_wind ($m\ s^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/06-30_-_07-02_2022/u_component_of_wind_ERA5_06-30_-_07-02_2022.nc
u_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 23) 



Running Calculations: 100%|██████████| 1/1 [00:09<00:00,  9.19s/it]


In [85]:
###########################
#DATE THREE (INTERESTING CASE)

#date information
date_string = "08-11 - 08-13 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/1 [00:00<?, ?it/s]/glade/derecho/scratch/aroseman/tmp/ipykernel_4486/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 1 Variables 

1. u_component_of_wind ($m\ s^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/../DATA/ERA5_Data/08-11_-_08-13_2022/u_component_of_wind_ERA5_08-11_-_08-13_2022.nc
u_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 23) 



Running Calculations: 100%|██████████| 1/1 [00:09<00:00,  9.73s/it]


In [44]:
##########################################################
# DOWNLOADING PRECIP CAMPAIGN DATA
##########################################################

In [52]:
# coorindates information
# Data BOUNDING BOX centered at Hsinchu, Taiwan PRECIP Campaign S-Pol radar moments data collected during the Prediction of Rainfall Extremes Campaign In the Pacific (PRECIP)
# (24.82N, 120.91E)
outputDirectory=SetOutputDirectory(campaign="PRECIP")
dataDirectory=SetDataDirectory(campaign="PRECIP")
#UTC OFFSET
UTC_offset="+8"

In [53]:
###########################
#DATE ONE (DRY CASE)

#date information
date_string = "06-05 - 06-07 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/10 [00:00<?, ?it/s]/glade/derecho/scratch/aroseman/tmp/ipykernel_32876/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 10 Variables 

1. u_component_of_wind ($m\ s^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/PRECIP/06-05_-_06-07_2022/u_component_of_wind_ERA5_06-05_-_06-07_2022.nc
u_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 22) 



Running Calculations:  10%|█         | 1/10 [00:09<01:22,  9.11s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_32876/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 10 Variables 

2. v_component_of_wind ($m\ s^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/PRECIP/06-05_-_06-07_2022/v_component_of_wind_ERA5_06-05_-_06-07_2022.nc
v_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 22) 



Running Calculations:  20%|██        | 2/10 [00:19<01:21, 10.14s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_32876/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


...

Running Calculations:  80%|████████  | 8/10 [01:23<00:23, 11.76s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_32876/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 10 Variables 

9. specific_cloud_ice_water_content ($g\ kg^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/PRECIP/06-05_-_06-07_2022/specific_cloud_ice_water_content_ERA5_06-05_-_06-07_2022.nc
specific_cloud_ice_water_content:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 22) 



Running Calculations:  90%|█████████ | 9/10 [01:40<00:13, 13.39s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_32876/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 10 Variables 

10. specific_rain_water_content ($g\ kg^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/PRECIP/06-05_-_06-07_2022/specific_rain_water_content_ERA5_06-05_-_06-07_2022.nc
specific_rain_water_content:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 22) 



Running Calculations: 100%|██████████| 10/10 [01:57<00:00, 11.72s/it]


In [51]:
###########################
#DATE TWO (MOIST CASE)

#date information
date_string = "07-16 - 07-18 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/10 [00:00<?, ?it/s]/glade/derecho/scratch/aroseman/tmp/ipykernel_32876/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 10 Variables 

1. u_component_of_wind ($m\ s^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/PRECIP/07-16_-_07-18_2022/u_component_of_wind_ERA5_07-16_-_07-18_2022.nc
u_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 22) 



Running Calculations:  10%|█         | 1/10 [00:09<01:28,  9.81s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_32876/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 10 Variables 

2. v_component_of_wind ($m\ s^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/PRECIP/07-16_-_07-18_2022/v_component_of_wind_ERA5_07-16_-_07-18_2022.nc
v_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 22) 



Running Calculations:  20%|██        | 2/10 [00:18<01:15,  9.43s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_32876/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


...

Running Calculations:  80%|████████  | 8/10 [01:21<00:23, 11.69s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_32876/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 10 Variables 

9. specific_cloud_ice_water_content ($g\ kg^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/PRECIP/07-16_-_07-18_2022/specific_cloud_ice_water_content_ERA5_07-16_-_07-18_2022.nc
specific_cloud_ice_water_content:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 22) 



Running Calculations:  90%|█████████ | 9/10 [01:38<00:13, 13.18s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_32876/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 10 Variables 

10. specific_rain_water_content ($g\ kg^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/PRECIP/07-16_-_07-18_2022/specific_rain_water_content_ERA5_07-16_-_07-18_2022.nc
specific_rain_water_content:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 22) 



Running Calculations: 100%|██████████| 10/10 [01:55<00:00, 11.57s/it]


In [34]:
##########################################################
# DOWNLOADING Hawaii DATA
##########################################################

In [35]:
# coorindates information
# GETTING BOUNDING BOX centered at Next Generation Weather Radar (NEXRAD) located on Molokai Island, Hawaii
# (21.133N, 157.180W) 
outputDirectory=SetOutputDirectory(campaign="Hawaii")
dataDirectory=SetDataDirectory(campaign="Hawaii")
#UTC OFFSET
UTC_offset="-10"

In [18]:
###########################
#DATE ONE (DRY CASE)

#date information
date_string = "08-07 - 08-09 (2022)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/12 [00:00<?, ?it/s]

Plotting 12 Variables 

1. u_component_of_wind ($m\ s^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/Hawaii/08-07_-_08-09_2022/u_component_of_wind_ERA5_08-07_-_08-09_2022.nc


/glade/derecho/scratch/aroseman/tmp/ipykernel_45946/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


u_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 21) 



Running Calculations:   8%|▊         | 1/12 [00:18<03:20, 18.25s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_45946/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


...

Running Calculations:  83%|████████▎ | 10/12 [01:56<00:27, 13.55s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_45946/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 12 Variables 

11. relative_humidity ($\%$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/Hawaii/08-07_-_08-09_2022/relative_humidity_ERA5_08-07_-_08-09_2022.nc
relative_humidity:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 21) 



Running Calculations:  92%|█████████▏| 11/12 [02:04<00:11, 11.91s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_45946/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 12 Variables 

12. geopotential ($m^{2}\ s^{-2}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/Hawaii/08-07_-_08-09_2022/geopotential_ERA5_08-07_-_08-09_2022.nc
geopotential:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 21) 



Running Calculations: 100%|██████████| 12/12 [02:13<00:00, 11.11s/it]


In [36]:
###########################
#DATE TWO (MOIST CASE)

#date information
date_string = "12-05 - 12-07 (2021)"
date_folder = MakeDateFolder(date_string)
loadDirectorys=GetLoadDirectory(dataDirectory,date_folder); #print(loadDirectorys)

RunAllCalculations(loadDirectorys, variables, date_string, date_folder, UTC_offset, outputDirectory, calculation, plotting)

Running Calculations:   0%|          | 0/12 [00:00<?, ?it/s]/glade/derecho/scratch/aroseman/tmp/ipykernel_45946/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 12 Variables 

1. u_component_of_wind ($m\ s^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/Hawaii/12-05_-_12-07_2021/u_component_of_wind_ERA5_12-05_-_12-07_2021.nc
u_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 21) 



Running Calculations:   8%|▊         | 1/12 [00:10<01:52, 10.23s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_45946/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 12 Variables 

2. v_component_of_wind ($m\ s^{-1}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/Hawaii/12-05_-_12-07_2021/v_component_of_wind_ERA5_12-05_-_12-07_2021.nc
v_component_of_wind:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 21) 



Running Calculations:  17%|█▋        | 2/12 [00:19<01:34,  9.46s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_45946/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


...

Running Calculations:  83%|████████▎ | 10/12 [01:48<00:26, 13.17s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_45946/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 12 Variables 

11. relative_humidity ($\%$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/Hawaii/12-05_-_12-07_2021/relative_humidity_ERA5_12-05_-_12-07_2021.nc
relative_humidity:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 21) 



Running Calculations:  92%|█████████▏| 11/12 [01:56<00:11, 11.66s/it]/glade/derecho/scratch/aroseman/tmp/ipykernel_45946/1409341371.py:66: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  numerics = Numerics(Nt=ncFile.dims['valid_time'],Np=ncFile.dims['pressure_level'],Nlat=ncFile.dims['latitude'],Nlon=ncFile.dims['longitude'], dt=60**2,


Plotting 12 Variables 

12. geopotential ($m^{2}\ s^{-2}$) → /glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/DATA/ERA5_Data/Hawaii/12-05_-_12-07_2021/geopotential_ERA5_12-05_-_12-07_2021.nc
geopotential:
 	(Nt, Np, Nlat, Nlon) =  (72, 32, 20, 21) 



Running Calculations: 100%|██████████| 12/12 [02:05<00:00, 10.45s/it]


In [ ]:
#####################################################

In [ ]:
#####################################################

In [56]:
#*#* Some Possible Future Improvements #*#*
#1. All Plots: x- and y-ticks somewhat incomplete
#2. Add U/V quivers